# DEH 30-Day PySpark Challenge
## Day 9 — when / otherwise: Conditional Logic

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows: {orders_df.count()}")

## Step 5 — Single condition with when() / otherwise()

In [ ]:
# Flag high value orders
orders_df = orders_df.withColumn(
    "is_high_value",
    F.when(F.col("unit_price") > 500, "Yes").otherwise("No")
)

orders_df.select("order_id", "unit_price", "is_high_value").show(5)

## Step 6 — Multiple conditions — chaining when()

In [ ]:
# Price tier — multiple chained conditions
orders_df = orders_df.withColumn(
    "price_tier",
    F.when(F.col("unit_price") > 500,  "Premium")
     .when(F.col("unit_price") > 200,  "Standard")
     .when(F.col("unit_price") > 50,   "Budget")
     .otherwise("Economy")
)

orders_df.select("order_id", "unit_price", "price_tier").show(8)

## Step 7 — What happens without otherwise()

In [ ]:
# No otherwise() — unmatched rows get null
orders_df.withColumn(
    "big_order",
    F.when(F.col("unit_price") > 1000, "Big Order")
     .when(F.col("unit_price") > 500,  "Medium Order")
    # no otherwise — rows under 500 become null
).select("order_id", "unit_price", "big_order").show(5)

## Step 8 — Compound conditions in when()

In [ ]:
# AND / OR inside when()
orders_df = orders_df.withColumn(
    "priority",
    F.when(
        (F.col("unit_price") > 500) & (F.col("status") == "Delivered"),
        "High Priority"
    ).when(
        (F.col("unit_price") > 200) | (F.col("region") == "East"),
        "Medium Priority"
    ).otherwise("Standard")
)

orders_df.select("order_id", "unit_price", "status", "region", "priority").show(5)

## Step 9 — when() returning a column expression

In [ ]:
# Return column expressions — different discount per region
orders_df = orders_df.withColumn(
    "final_price",
    F.when(F.col("region") == "East", F.col("unit_price") * 0.9)
     .when(F.col("region") == "West", F.col("unit_price") * 0.85)
     .otherwise(F.col("unit_price"))
)

orders_df.select("order_id", "region", "unit_price", "final_price").show(5)

## Step 10 — when() inside select()

In [ ]:
# when() works inside select() too
orders_df.select(
    F.col("order_id"),
    F.col("unit_price"),
    F.when(F.col("unit_price") > 500, "Premium")
     .when(F.col("unit_price") > 100, "Standard")
     .otherwise("Economy").alias("tier")
).show(5)

---
## Assignment — Day 9

---

### Task 1
Add a column `order_size` using `when()`:  
- quantity >= 4 → "Large"  
- quantity >= 2 → "Medium"  
- otherwise → "Small"  

Show `order_id`, `quantity`, and `order_size`.

In [ ]:
# Task 1 — Write your code here


### Task 2
Add a column `discount_label`:  
- discount_pct == 0 → "No Discount"  
- discount_pct <= 10 → "Low Discount"  
- discount_pct <= 15 → "Medium Discount"  
- otherwise → "High Discount"  

Show the distribution using `groupBy("discount_label").count()`.

In [ ]:
# Task 2 — Write your code here


### Task 3
Add a column `vip_order`:  
- True when unit_price > 500 AND status == "Delivered"  
- False otherwise  

How many VIP orders are there?

In [ ]:
# Task 3 — Write your code here


### Task 4
Add a `regional_price` column:  
- East region → 10% off  
- West region → 15% off  
- All other regions → full price  

Use `when()` returning a column expression.  
Show `order_id`, `region`, `unit_price`, and `regional_price`.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*